In [1]:
# ── IMPORTS ─────────────────────────────────────────────────
import pandas as pd
import numpy as np


# ============================================================
# ETAPA A — EXTRAÇÃO
# ============================================================

df = pd.read_csv('grupo2_clinica.csv')

print("Shape:", df.shape)
print()
df.head()

Shape: (2860, 13)



,id_atendimento,data_atendimento,id_paciente,nome_paciente,idade,sexo,especialidade,medico,convenio,diagnostico,valor_consulta,retorno,duracao_minutos
0,ATD-5000,01.01.2025,PAC-729,Maya Nunes,64.0,M,Neurologia,Dr. Novais,SulAmérica,Hipertensão,438.91,S,18
1,ATD-5001,09/02/2024,PAC-1842,Ian Guerra,31.0,NaN,Ginecologia,Dr. Vargas,unimed,Lombalgia,539.90,Não,41
2,ATD-5002,20/11/2024,PAC-2607,Ana Sophia Fernandes,64.0,Feminino,Ortopédia,Dr. Casa Grande,Amil,Hipertensão,555.54,nao,38
3,ATD-5003,31.01.2024,PAC-2593,Maysa Mendes,23.0,NaN,Clínica Geral,Dr. Mendes,UNIMED,Lombalgia,391.07,N,300
4,ATD-5004,23.04.2024,PAC-2538,Henry Gabriel da Mata,16.0,Masculino,Ortopedia,Dr. Costela,Particular,Enxaqueca,521.33,N,36


In [3]:
# Inspeção dos tipos de dados
df.dtypes

,0
id_atendimento,object
data_atendimento,object
id_paciente,object
nome_paciente,object
idade,float64
sexo,object
especialidade,object
medico,object
convenio,object
diagnostico,object


In [4]:
# Contagem de valores nulos por coluna
print("=== Valores Nulos ===")
print(df.isnull().sum())

=== Valores Nulos ===
id_atendimento        0
data_atendimento      0
id_paciente           0
nome_paciente         0
idade                35
sexo                386
especialidade         0
medico                0
convenio              0
diagnostico         939
valor_consulta       77
retorno             370
duracao_minutos       0
dtype: int64


In [5]:
# Verificação de duplicatas
print("=== Duplicatas ===")
print(df.duplicated().sum())

=== Duplicatas ===
60


In [6]:
# Análise de problemas específicos antes da limpeza
print("Convênios únicos:", df['convenio'].unique())
print()
print("Especialidades únicas:", df['especialidade'].unique())
print()
print("Retorno únicos:", df['retorno'].unique())
print()
print("Sexo únicos:", df['sexo'].unique())
print()
print("Idades impossíveis (< 0 ou > 120):", df[(df['idade'] < 0) | (df['idade'] > 120)].shape[0])
print("Min / Max idade:", df['idade'].min(), "/", df['idade'].max())
print()
print("Formatos de data (amostra):", df['data_atendimento'].head(10).tolist())

Convênios únicos: ['SulAmérica' 'unimed' 'Amil' 'UNIMED' 'Particular' 'Bradesco Saúde'
 'Unimed' 'particular']

Especialidades únicas: ['Neurologia' 'Ginecologia' 'Ortopédia' 'Clínica Geral' 'Ortopedia'
 'Pediatria' 'CARDIOLOGIA' 'Cardiologia' 'neurologia' 'Gineco'
 'clinica geral' 'Pediatria ']

Retorno únicos: ['S' 'Não' 'nao' 'N' 'sim' nan 'Sim']

Sexo únicos: ['M' nan 'Feminino' 'Masculino' 'f' 'F' 'm']

Idades impossíveis (< 0 ou > 120): 69
Min / Max idade: -5.0 / 999.0

Formatos de data (amostra): ['01.01.2025', '09/02/2024', '20/11/2024', '31.01.2024', '23.04.2024', '10-07-2023', '2025-03-01', '01-23-2026', '12/06/2024', '2025-06-07']


In [7]:
# Estatísticas descritivas das colunas numéricas
df[['idade', 'valor_consulta', 'duracao_minutos']].describe()

,idade,valor_consulta,duracao_minutos
count,2825.000000,2783.000000,2860.000000
mean,52.286726,339.255713,37.777273
std,86.394735,149.691470,35.290287
min,-5.000000,80.210000,-10.000000
25%,22.000000,208.855000,22.000000
50%,45.000000,339.130000,35.000000
75%,68.000000,470.655000,48.000000
max,999.000000,599.940000,300.000000


In [8]:
# ============================================================
# ETAPA B — LIMPEZA
# ============================================================

# ── B1. Remover duplicatas ───────────────────────────────────
# Impacto se não tratar: 60 atendimentos fantasmas inflariam
# contagem total e receita calculada no dashboard.

antes = df.shape[0]
df.drop_duplicates(inplace=True)
depois = df.shape[0]
print(f"Duplicatas removidas: {antes - depois} | Registros restantes: {depois}")

Duplicatas removidas: 60 | Registros restantes: 2800


In [9]:
# ── B2. Remover idades inválidas ─────────────────────────────
# Impacto se não tratar: pacientes com idade -5 ou 999 seriam
# classificados em faixas etárias erradas, distorcendo o gráfico
# principal que responde a Pergunta 1.

antes = df.shape[0]
df = df[(df['idade'] >= 0) & (df['idade'] <= 120)]
depois = df.shape[0]
print(f"Registros com idades inválidas removidos: {antes - depois} | Restantes: {depois}")

Registros com idades inválidas removidos: 103 | Restantes: 2697


In [10]:
# ── B3. Preencher nulos de idade com mediana ─────────────────
# Usamos a mediana porque ela é menos sensível a outliers que
# a média. Com idades de 0 a 120, a mediana representa melhor
# o valor central da distribuição.

print("Nulos em 'idade' antes:", df['idade'].isnull().sum())
df['idade'] = df['idade'].fillna(df['idade'].median())
print("Nulos em 'idade' depois:", df['idade'].isnull().sum())

Nulos em 'idade' antes: 0
Nulos em 'idade' depois: 0


In [11]:
# ── B4. Padronizar convênios ─────────────────────────────────
# Impacto se não tratar: 'unimed', 'UNIMED' e 'Unimed' aparecem
# como 3 convênios diferentes. A receita da Unimed — que é o
# maior convênio — ficaria fragmentada em 3 barras no dashboard,
# impossibilitando responder corretamente a Pergunta 2.

print("Antes:", df['convenio'].value_counts().to_dict())

df['convenio'] = df['convenio'].str.strip().str.lower()
df['convenio'] = df['convenio'].replace({
    'unimed'        : 'Unimed',
    'sulamerica'    : 'SulAmérica',
    'sulamérica'    : 'SulAmérica',
    'amil'          : 'Amil',
    'bradesco saúde': 'Bradesco Saúde',
    'particular'    : 'Particular'
})

print("\nDepois:", df['convenio'].value_counts().to_dict())

Antes: {'UNIMED': 357, 'unimed': 349, 'SulAmérica': 345, 'Particular': 343, 'particular': 337, 'Bradesco Saúde': 325, 'Amil': 322, 'Unimed': 319}

Depois: {'Unimed': 1025, 'Particular': 680, 'SulAmérica': 345, 'Bradesco Saúde': 325, 'Amil': 322}


In [12]:
# ── B5. Padronizar especialidades ────────────────────────────
# Impacto se não tratar: 'CARDIOLOGIA' e 'Cardiologia' seriam
# duas especialidades distintas no gráfico, dividindo os
# atendimentos de Cardiologia em duas barras separadas.

print("Antes:", df['especialidade'].value_counts().to_dict())

df['especialidade'] = df['especialidade'].str.strip()
df['especialidade'] = df['especialidade'].replace({
    'Ortopedia'   : 'Ortopédia',
    'CARDIOLOGIA' : 'Cardiologia',
    'neurologia'  : 'Neurologia',
    'Gineco'      : 'Ginecologia',
    'clinica geral': 'Clínica Geral',
    'Pediatria '  : 'Pediatria'
})

print("\nDepois:", df['especialidade'].value_counts().to_dict())

Antes: {'Ortopedia': 382, 'Clínica Geral': 381, 'Pediatria': 381, 'Ginecologia': 380, 'Neurologia': 376, 'Cardiologia': 364, 'clinica geral': 90, 'neurologia': 82, 'Pediatria ': 75, 'Ortopédia': 70, 'Gineco': 68, 'CARDIOLOGIA': 48}

Depois: {'Clínica Geral': 471, 'Neurologia': 458, 'Pediatria': 456, 'Ortopédia': 452, 'Ginecologia': 448, 'Cardiologia': 412}


In [13]:
# ── B6. Padronizar campo retorno ─────────────────────────────
# Impacto se não tratar: 'S', 'Sim', 'sim', 'N', 'nao' e NaN
# impossibilitam calcular a taxa de retorno corretamente.

print("Antes:", df['retorno'].value_counts(dropna=False).to_dict())

df['retorno'] = df['retorno'].str.strip().str.lower()
df['retorno'] = df['retorno'].replace({
    's'  : 'Sim',
    'sim': 'Sim',
    'n'  : 'Não',
    'nao': 'Não',
    'não': 'Não'
})
df['retorno'] = df['retorno'].fillna('Não informado')

print("\nDepois:", df['retorno'].value_counts().to_dict())

Antes: {'Sim': 418, 'N': 412, 'S': 403, 'nao': 402, 'Não': 368, nan: 356, 'sim': 338}

Depois: {'Não': 1182, 'Sim': 1159, 'Não informado': 356}


In [14]:
# ── B7. Padronizar campo sexo ────────────────────────────────
# 'M', 'm', 'Masculino', 'F', 'f', 'Feminino' — 6 grafias para
# 2 categorias. Padronizamos para Masculino / Feminino.

print("Antes:", df['sexo'].value_counts(dropna=False).to_dict())

df['sexo'] = df['sexo'].str.strip()
df['sexo'] = df['sexo'].replace({
    'M': 'Masculino',
    'm': 'Masculino',
    'F': 'Feminino',
    'f': 'Feminino'
})

print("\nDepois:", df['sexo'].value_counts(dropna=False).to_dict())

Antes: {'Masculino': 400, 'M': 397, 'f': 386, 'Feminino': 385, 'm': 378, 'F': 377, nan: 374}

Depois: {'Masculino': 1175, 'Feminino': 1148, nan: 374}


In [15]:
# ── B8. Preencher nulos de valor_consulta com mediana ────────
# Impacto se não tratar: 77 consultas sem valor subestimam a
# receita total da clínica no KPI do dashboard.
# Usamos a mediana pelo mesmo motivo da idade (robustez a outliers).

print("Nulos em 'valor_consulta' antes:", df['valor_consulta'].isnull().sum())
df['valor_consulta'] = df['valor_consulta'].fillna(df['valor_consulta'].median())
print("Nulos em 'valor_consulta' depois:", df['valor_consulta'].isnull().sum())

Nulos em 'valor_consulta' antes: 73
Nulos em 'valor_consulta' depois: 0


In [16]:
# ── B9. Converter datas (5 formatos diferentes) ───────────────
# Impacto se não tratar: o campo permanece como texto (string),
# tornando impossível extrair o mês e criar o gráfico de
# atendimentos por mês no Looker Studio.
#
# Formatos identificados:
#   dd.mm.aaaa | dd/mm/aaaa | aaaa-mm-dd | dd-mm-aaaa | mm-dd-aaaa

def parse_date(d):
    for fmt in ('%d.%m.%Y', '%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%m-%d-%Y'):
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT

print("Tipo antes:", df['data_atendimento'].dtype)
df['data_atendimento'] = df['data_atendimento'].apply(parse_date)
print("Tipo depois:", df['data_atendimento'].dtype)
print("Datas nulas após conversão:", df['data_atendimento'].isnull().sum())

Tipo antes: object
Tipo depois: datetime64[ns]
Datas nulas após conversão: 0


In [17]:
# ============================================================
# ETAPA C — ENRIQUECIMENTO
# Mínimo de 2 colunas calculadas exigido pelo professor.
# Criamos 3, todas relevantes para as perguntas de negócio.
# ============================================================

# ── C1. Faixa Etária ─────────────────────────────────────────
# Responde diretamente a Pergunta 1.
# Classificação baseada em critérios do Estatuto da Criança e do
# Adolescente (ECA) e do Estatuto do Idoso (Lei 10.741/2003).

def faixa_etaria(idade):
    if idade <= 12:
        return 'Criança'
    elif idade <= 17:
        return 'Adolescente'
    elif idade <= 59:
        return 'Adulto'
    else:
        return 'Idoso'

df['faixa_etaria'] = df['idade'].apply(faixa_etaria)
print("=== Atendimentos por Faixa Etária ===")
print(df['faixa_etaria'].value_counts())

=== Atendimentos por Faixa Etária ===
faixa_etaria
Adulto         1265
Idoso           913
Criança         382
Adolescente     137
Name: count, dtype: int64


In [18]:
# ── C2. Mês de Atendimento ───────────────────────────────────
# Permite identificar sazonalidade (meses de pico de demanda)
# no gráfico de linha do dashboard.

df['mes_atendimento'] = df['data_atendimento'].dt.strftime('%B')
print("=== Atendimentos por Mês ===")
print(df['mes_atendimento'].value_counts())

=== Atendimentos por Mês ===
mes_atendimento
March        255
August       248
October      239
December     235
January      232
May          222
November     220
February     213
September    211
July         211
June         210
April        201
Name: count, dtype: int64


In [19]:
# ── C3. Categoria de Duração ─────────────────────────────────
# Classifica consultas em 3 categorias para análise operacional.
# Útil para identificar especialidades com consultas mais longas.

def categoria_duracao(minutos):
    if minutos < 20:
        return 'Curta'
    elif minutos <= 40:
        return 'Padrão'
    else:
        return 'Longa'

df['categoria_duracao'] = df['duracao_minutos'].apply(categoria_duracao)
print("=== Categoria de Duração ===")
print(df['categoria_duracao'].value_counts())

=== Categoria de Duração ===
categoria_duracao
Padrão    1091
Longa     1041
Curta      565
Name: count, dtype: int64


In [20]:
# ============================================================
# ANÁLISE — RESPONDENDO ÀS PERGUNTAS DE NEGÓCIO
# ============================================================

# PERGUNTA 1: Qual faixa etária concentra mais atendimentos?
print("=" * 50)
print("PERGUNTA 1 — Faixa etária com mais atendimentos")
print("=" * 50)
resultado_p1 = df['faixa_etaria'].value_counts()
print(resultado_p1)
print()
print(">>> Resposta:", resultado_p1.idxmax(),
      f"com {resultado_p1.max()} atendimentos",
      f"({resultado_p1.max() / len(df) * 100:.1f}% do total)")

PERGUNTA 1 — Faixa etária com mais atendimentos
faixa_etaria
Adulto         1265
Idoso           913
Criança         382
Adolescente     137
Name: count, dtype: int64

>>> Resposta: Adulto com 1265 atendimentos (46.9% do total)


In [21]:
# PERGUNTA 2: Qual convênio gera mais receita?
print("=" * 50)
print("PERGUNTA 2 — Convênio com maior receita")
print("=" * 50)
receita_convenio = (
    df.groupby('convenio')['valor_consulta']
    .sum()
    .sort_values(ascending=False)
)
print(receita_convenio.apply(lambda x: f"R$ {x:,.2f}"))
print()
print(">>> Resposta:", receita_convenio.idxmax(),
      f"— R$ {receita_convenio.max():,.2f}")

PERGUNTA 2 — Convênio com maior receita
convenio
Unimed            R$ 353,752.86
Particular        R$ 224,989.44
SulAmérica        R$ 116,057.45
Bradesco Saúde    R$ 112,748.60
Amil              R$ 106,087.39
Name: valor_consulta, dtype: object

>>> Resposta: Unimed — R$ 353,752.86


In [22]:
# KPIs gerais do dashboard
print("=" * 50)
print("KPIs GERAIS")
print("=" * 50)
print(f"Total de Atendimentos : {len(df):,}")
print(f"Receita Total         : R$ {df['valor_consulta'].sum():,.2f}")
print(f"Ticket Médio          : R$ {df['valor_consulta'].mean():,.2f}")
retornos = df[df['retorno'] == 'Sim'].shape[0]
print(f"Taxa de Retorno       : {retornos / len(df) * 100:.1f}%")

KPIs GERAIS
Total de Atendimentos : 2,697
Receita Total         : R$ 913,635.73
Ticket Médio          : R$ 338.76
Taxa de Retorno       : 43.0%


In [23]:
# ============================================================
# ETAPA D — EXPORTAÇÃO
# ============================================================

df.to_csv('dataset_limpo.csv', index=False, encoding='utf-8-sig')

print("Arquivo 'dataset_limpo.csv' exportado com sucesso!")
print("Shape final:", df.shape)
print("Colunas:", df.columns.tolist())

Arquivo 'dataset_limpo.csv' exportado com sucesso!
Shape final: (2697, 16)
Colunas: ['id_atendimento', 'data_atendimento', 'id_paciente', 'nome_paciente', 'idade', 'sexo', 'especialidade', 'medico', 'convenio', 'diagnostico', 'valor_consulta', 'retorno', 'duracao_minutos', 'faixa_etaria', 'mes_atendimento', 'categoria_duracao']
